In [ ]:
import pandas as pd
import numpy as np

In [ ]:
pantheon = pd.read_csv("../data/pantheon.tsv", sep="\t")
fcdo = pd.read_csv(
    "../data/FCDO_Geographical_Names_Index_March_2024.csv", encoding_errors="ignore"
)
cv = pd.read_csv(
    "../data/cross-verified-database.csv",
    usecols=["name", "wiki_readers_2015_2018"],
    encoding_errors="ignore",
)
# cf_full = pd.read_csv("cross-verified-database.csv", encoding_errors="ignore")

In [ ]:
names = cv
names["name"] = cv["name"].str.lower()
names[["first", "last"]] = names.name.str.rsplit("_", n=1, expand=True)

In [ ]:
countries = fcdo["Name"]
countries = countries.str.lower()

In [ ]:
def add_every_letter(text):
    result = []
    for i in range(len(text) + 1):
        for char in "abcdefghijklmnopqrstuvwxyz":
            new_text = text[:i] + char + text[i:]
            result.append(new_text)
    return result

In [ ]:
result = [add_every_letter(x) for x in countries]
df = pd.DataFrame(result)
countries_permutations = pd.concat([df[col] for col in df.columns])

In [ ]:
test = names["first"].isin(countries_permutations)

In [ ]:
first = countries_permutations.isin(names["first"])
first = countries_permutations[first]
first = pd.DataFrame(first).rename(columns={0: "first_name"})
# first["match_first"] = 'Y'
print(first)

In [ ]:
last = countries_permutations.isin(names["last"])
last = countries_permutations[last]
last = pd.DataFrame(last).rename(columns={0: "last_name"}).dropna()
# last["match_last"] = 'Y'
print(last)

In [ ]:
df_merge = names.merge(first, left_on="first", right_on="first_name", how="left")
df_merge = df_merge.merge(last, left_on="last", right_on="last_name", how="left")

In [ ]:
final_prep = df_merge
final_prep["match"] = None
final_prep.loc[
    final_prep["first_name"].notna() | final_prep["last_name"].notna(), "match"
] = "Y"
final = final_prep.dropna(subset=["match"])